# 03 · Preparación de datos y prevención de *data leakage*
Guía 16.4 y 16.5. Salida: `data/processed/split.csv` y `classes.json` (una sola vez; todos los experimentos los leen), más `src/extras.py` con utilidades para los notebooks 04 y 05.

**Pregunta de la guía:** ¿qué información tenía el modelo en el momento real de predecir? En este proyecto la unidad de agrupación es la **foto de origen** (varios recortes de una misma foto no pueden repartirse entre train y test) y, además, no puede haber casi-duplicados entre particiones.

**Qué cambió respecto a la plantilla original:** la limpieza, la agrupación y la partición ya se hicieron y se justificaron en el notebook 01 (splits oficiales de BRACOL, casi-duplicados resueltos dando prioridad a test). Aquí se **verifican de forma independiente**, se congela la partición y se define el preprocesamiento.

## 0 · Entorno y carga

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys, os, zipfile, json, random
PROJECT = '/content/drive/MyDrive/proyecto_cafe_cnn'
os.environ['PROJECT_DIR'] = PROJECT
sys.path.insert(0, f'{PROJECT}/src')
import common as C
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
P = C.paths()
SEED = 42
C.set_seed(SEED)
plt.rcParams.update({'font.family': 'serif', 'figure.dpi': 120})

# Recortes de BRACOL en disco local (si es una sesión nueva)
OUT = '/content/bracol_crops'
if not os.path.exists(OUT):
    with zipfile.ZipFile(f"{P['data/processed']}/bracol_crops.zip") as z:
        z.extractall(OUT)
    print('Recortes restaurados desde Drive')

df = pd.read_csv(f"{P['data/processed']}/manifest_final.csv")
assert df.path.map(os.path.exists).all(), 'Hay rutas del manifest que no existen'
print(df.groupby(['source', 'split']).size().unstack(fill_value=0))

## 1 · Decisiones de partición (ya aplicadas en el notebook 01)
| Decisión | Detalle | Evidencia |
|---|---|---|
| BRACOL: splits oficiales | train/val/test de la versión revisada de Roboflow; no comparten nombres base | notebook 01, sección 1 |
| Unidad de agrupación | foto de origen; tras depurar, grupo = casi-duplicado (dHash ≤ 4) | notebook 01, sección 3 |
| Casi-duplicados entre splits | se conserva el recorte en la partición de mayor prioridad (test > val > train) | `casi_duplicados_eliminados` en `calidad_datos.csv` |
| JMuBEN | solo validación externa (500 por clase); nunca entrena ni se usa para seleccionar | notebook 01, sección 4 |
| `Healthy` | solo existe en JMuBEN: se conserva en el archivo pero no entra en el entrenamiento ni en la métrica externa de 4 clases | notebook 02 |

## 2 · Clases y estratificación

In [ ]:
CLASSES = sorted(df[df.source == 'bracol'].label.unique())
print('Clases de entrenamiento:', CLASSES)
print('Clases solo en JMuBEN (excluidas de la métrica externa):', sorted(set(df.label) - set(CLASSES)))

t = df.groupby(['split', 'label']).size().unstack(fill_value=0)
display(t)
prop = (t.loc[['train', 'val', 'test'], CLASSES].T / t.loc[['train', 'val', 'test'], CLASSES].sum(axis=1)).T.round(3)
print('Proporción de cada clase dentro de cada split (BRACOL):')
display(prop)

## 3 · Verificación independiente de fuga

In [ ]:
# 1) Entre train/val/test: md5 y grupo (debe imprimir 0 en todo y True)
b = df[df.source == 'bracol']
ok = C.check_leakage(b, cols=('md5', 'group'))
print('Sin fuga entre train/val/test:', ok); assert ok

# 2) Externo vs BRACOL: md5 idénticos
ext = df[df.split == 'external']
print('md5 compartidos externo ∩ BRACOL:', len(set(ext.md5) & set(b.md5)))
assert not set(ext.md5) & set(b.md5)

# 3) Externo vs BRACOL: casi-duplicados por dHash (distancia de Hamming <= 4)
A = np.array([[int(c) for c in s] for s in ext.dhash], dtype=np.float32)
B = np.array([[int(c) for c in s] for s in b.dhash], dtype=np.float32)
dist = A.shape[1] - (A @ B.T + (1 - A) @ (1 - B).T)
n_near = int((dist.min(axis=1) <= 4).sum())
print(f'imágenes externas con casi-duplicado en BRACOL: {n_near} de {len(ext)}')

In [ ]:
# Si hubo casi-duplicados externos, se retiran del conjunto externo (nunca de BRACOL)
if n_near:
    quitar = ext.index[dist.min(axis=1) <= 4]
    df = df.drop(index=quitar).reset_index(drop=True)
    print('retirados del externo:', n_near)
print('Conjunto externo final:', (df.split == 'external').sum(), 'imágenes')

## 4 · Congelar la partición

In [ ]:
df.drop(columns=['dhash']).to_csv(f"{P['data/processed']}/split.csv", index=False)
json.dump(CLASSES, open(f"{P['data/processed']}/classes.json", 'w'))
print('Guardado split.csv con', len(df), 'filas y classes.json con', CLASSES)

# Fig. · Distribución final por partición
t = df.groupby(['split', 'label']).size().unstack(0).fillna(0).astype(int)[['train', 'val', 'test', 'external']]
ax = t.plot(kind='bar', figsize=(9, 4), edgecolor='black', rot=0, logy=True)
ax.set_ylabel('N.º de muestras (escala log)'); ax.set_xlabel(''); ax.grid(axis='y', ls='--', alpha=.5)
ax.set_title('Partición final'); plt.tight_layout()
plt.savefig(f"{P['figures']}/particion_final.png", dpi=300); plt.show(); display(t)

## 5 · Preprocesamiento y aumento de datos
- **Entrada:** `C.make_dataset` carga cada imagen, la redimensiona a 224×224 (bilineal) y devuelve valores en [0, 255]; la normalización (`Rescaling` o `preprocess_input` de ResNet50) va **dentro del modelo**, idéntica en train, val y test.
- **Aumento solo en train:** volteos, rotación (±21.6°), zoom y contraste. Val y test nunca se transforman.
- **Opción A/B (degradación de resolución).** En el EDA se vio que JMuBEN mide 128×128 y se amplía, y que Rust en BRACOL es la clase de menor resolución. La opción consiste en reducir aleatoriamente la muestra (48–128 px) y volver a ampliarla, solo en train, para que el modelo no dependa de la nitidez. Se implementa **sin modificar `common.py`**, en `src/extras.py`, y se comparará contra la versión sin degradación en el notebook 04. **La decisión se toma con val, nunca con test ni con el conjunto externo.**

In [ ]:
%%writefile /content/drive/MyDrive/proyecto_cafe_cnn/src/extras.py
# extras.py - utilidades adicionales (no modifica common.py)
import os
import shutil
import zipfile
import numpy as np
import pandas as pd
import common as C

LOCAL_CROPS = '/content/bracol_crops'
LOCAL_JM = '/content/jmuben_sample'


def prepare_local():
    # Copia los datos a disco local (mucho más rápido que Drive) y devuelve el DataFrame de split.csv
    # con las rutas ya apuntando a /content.
    P = C.paths()
    if not os.path.exists(LOCAL_CROPS):
        with zipfile.ZipFile(f"{P['data/processed']}/bracol_crops.zip") as z:
            z.extractall(LOCAL_CROPS)
    src = f"{P['data/processed']}/jmuben_sample"
    if not os.path.exists(LOCAL_JM):
        shutil.copytree(src, LOCAL_JM)
    df = pd.read_csv(f"{P['data/processed']}/split.csv")
    df['path'] = df.path.str.replace(src, LOCAL_JM, regex=False)
    faltan = int((~df.path.map(os.path.exists)).sum())
    assert faltan == 0, f'{faltan} rutas no existen'
    return df


def degrade_batch(x, p=0.5, lo=48, hi=128):
    # Con probabilidad p, reduce cada imagen a un lado aleatorio en [lo, hi) y la vuelve a ampliar.
    import tensorflow as tf
    x = tf.convert_to_tensor(x, tf.float32)
    H, W = int(x.shape[1]), int(x.shape[2])

    def one(img):
        def do():
            s = tf.random.uniform([], lo, hi, dtype=tf.int32)
            small = tf.image.resize(img, tf.stack([s, s]), method='bilinear', antialias=True)
            return tf.image.resize(small, (H, W), method='bilinear')
        out = tf.cond(tf.random.uniform([]) < p, do, lambda: img)
        out.set_shape((H, W, 3))
        return out
    return tf.map_fn(one, x)


_orig_make_dataset = None


def enable_degrade(p=0.5):
    # Reemplaza C.make_dataset por una versión que degrada resolución solo con training=True.
    # run_experiment busca make_dataset en el módulo common, así que no hay que tocar nada más.
    global _orig_make_dataset
    if _orig_make_dataset is None:
        _orig_make_dataset = C.make_dataset

    def wrapped(file_paths, int_labels, batch_size=32, training=False, seed=42, img_size=C.IMG_SIZE):
        import tensorflow as tf
        ds = _orig_make_dataset(file_paths, int_labels, batch_size, training, seed, img_size)
        if training and p > 0:
            ds = ds.map(lambda x, y: (degrade_batch(x, p), y), num_parallel_calls=tf.data.AUTOTUNE)
            ds = ds.prefetch(tf.data.AUTOTUNE)
        return ds
    C.make_dataset = wrapped
    print(f'degradación de resolución ACTIVADA (p={p})')


def disable_degrade():
    global _orig_make_dataset
    if _orig_make_dataset is not None:
        C.make_dataset = _orig_make_dataset
        _orig_make_dataset = None
    print('degradación de resolución DESACTIVADA')

In [ ]:
# Demostración visual: aumento estándar vs. aumento + degradación (p=1 para verla siempre)
import importlib, extras
importlib.reload(extras)

def ejemplo(source, label):
    d = df[(df.source == source) & (df.label == label) & (df.split.isin(['train', 'external']))]
    return d.sample(1, random_state=SEED).path.iloc[0]

def lote(path, n=6):
    ds = C.make_dataset([path] * n, [0] * n, batch_size=n, training=True, seed=SEED)
    return next(iter(ds))[0].numpy()

casos = [('bracol', 'rust'), ('jmuben', 'rust')]
fig, axes = plt.subplots(4, 6, figsize=(13, 9))
for k, (s, l) in enumerate(casos):
    x = lote(ejemplo(s, l))
    xd = extras.degrade_batch(x, p=1.0).numpy()
    for j in range(6):
        axes[2*k, j].imshow(x[j].astype('uint8')); axes[2*k, j].axis('off')
        axes[2*k+1, j].imshow(xd[j].astype('uint8')); axes[2*k+1, j].axis('off')
    axes[2*k, 0].set_title(f'{s} · {l}: aumento estándar', loc='left', fontsize=10, fontweight='bold')
    axes[2*k+1, 0].set_title(f'{s} · {l}: + degradación de resolución', loc='left', fontsize=10, fontweight='bold')
plt.tight_layout(); plt.savefig(f"{P['figures']}/aumento_demo.png", dpi=300); plt.show()

In [ ]:
# Prueba de humo de prepare_local (la usarán los notebooks 04 y 05)
d_local = extras.prepare_local()
print(d_local.groupby(['source', 'split']).size().unstack(fill_value=0))
print(d_local.path.iloc[0]); print(d_local.path.iloc[-1])

## Justificación técnica de cada transformación (para el informe)
| Paso | Por qué |
|---|---|
| Recortes de caja en BRACOL | La lesión mide ~55 px en una imagen de 2048×1024; al redimensionar la foto completa quedaría de pocos píxeles |
| Eliminar corruptas / duplicados exactos | Unicidad; evita contar dos veces la misma evidencia |
| Casi-duplicados (dHash ≤ 4) con prioridad a test | Evita que el test contenga copias del entrenamiento sin tocar el test |
| Splits oficiales de BRACOL, agrupados | Mantiene proporciones de clase y evita separar recortes de una misma foto |
| JMuBEN solo externo | Mide generalización a otra procedencia; nunca participa en entrenamiento ni en selección |
| Redimensionar a 224×224 y normalizar dentro del modelo | Requisito de ResNet50; mismo preprocesamiento en train/val/test |
| Aumento solo en train | Val/test deben reflejar datos reales sin transformar |
| Degradación de resolución (opcional) | Reduce la dependencia de la nitidez, que difiere entre BRACOL y JMuBEN; se acepta solo si mejora en **val** |
| `class_weight` balanceado | Razón de desbalance ≈ 31:1 en train; no se modifica el dataset para corregirlo |

*No se modifica el dataset para mejorar métricas.*